<a href="https://colab.research.google.com/github/Wasin-Winyarat/baanbrew-dashborad/blob/main/Lab2_1_Customers_Profiling_Wasin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2.1 (ต่อ) · Data Profiling และทำความสะอาดข้อมูลลูกค้า

ใช้ขั้นตอนเดียวกับ `Lab2_1_Data_Cleaning.ipynb` แต่กับ `customers.xlsx` (โจทย์ "เสร็จเร็ว? ลองต่อ" ข้อสุดท้าย)

**ต่างจาก sales อย่างไร**
- คอลัมน์ต่างกัน: `customer_id, nickname, gender, age_group, home_branch_id, joined_date, phone` ไม่มี datetime / unit_price / qty จึงใช้ prompt ของ sales ตรง ๆ ไม่ได้
- ลูกค้า 1 คน = 1 แถว (ต่างจาก sales ที่ 1 บิลมีหลายแถว) จึงตรวจ `customer_id` ซ้ำได้โดยตรง
- ปัญหาที่ตรวจจากไฟล์เดียวไม่เจอ ต้องเทียบข้ามตาราง: สมัครก่อนสาขาบ้านเปิด (ใช้ `branches.xlsx`) และซื้อก่อนสมัคร / id ที่ไม่ตรงกับยอดขาย (ใช้ `sales.csv` ในขั้นที่ 5B)

| ขั้น | ทำอะไร |
|---|---|
| 0–1 | โหลดข้อมูล |
| 2 | Profiling: หาปัญหา **แต่ยังไม่แก้** |
| 3 | ตัดสินใจ |
| 4 | ทำความสะอาด (บน `clean` ห้ามแก้ `df`) |
| 5 | Verify |
| 5B | ตรวจข้ามตารางกับ `sales.csv` |
| 6 | Export |

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน ฟังก์ชัน `load_xlsx()` อ่านทุกคอลัมน์เป็นข้อความ และถ้าไม่มีไฟล์ใน Colab จะให้อัปโหลด

In [ ]:
import os, re
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

STD_GENDER = ["หญิง", "ชาย", "ไม่ระบุ"]
STD_AGE = ["ต่ำกว่า 18", "18-24", "25-34", "35-44", "45-54", "55+"]


def load_xlsx(name):
    """อ่าน Excel ทุกคอลัมน์เป็นข้อความ ค่าว่างเป็น "" ถ้าไม่มีไฟล์จะให้อัปโหลด (Google Colab)"""
    if not os.path.exists(name):
        try:
            from google.colab import files
            print(f"ไม่พบ {name} กรุณาอัปโหลดไฟล์")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"ไม่พบไฟล์ {name} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
    return pd.read_excel(name, dtype=str, keep_default_na=False)

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก
อัปโหลด `customers.xlsx` และ `branches.xlsx` จากโฟลเดอร์ `data/`

In [ ]:
df = load_xlsx("customers.xlsx")
branches = load_xlsx("branches.xlsx")
STD_BRANCH_ID = branches["branch_id"].tolist()

# วันเปิดสาขา (ตารางอ้างอิง ไม่ใช่ข้อมูลที่เรากำลังตรวจ) ใน Excel เก็บเป็นค่าวันที่
# อ่านด้วย dtype=str จึงได้ "2023-06-01 00:00:00" ตัดเหลือ 10 ตัวแรก = YYYY-MM-DD
OPENED = branches.set_index("branch_id")["opened_date"].str[:10]

print(f"{len(df):,} แถว × {df.shape[1]} คอลัมน์ · สาขา {len(branches)} สาขา")
df.head(8)

3,000 แถว × 7 คอลัมน์ · สาขา 5 สาขา


,customer_id,nickname,gender,age_group,home_branch_id,joined_date,phone
0,C00001,กมล,หญิง,18-24,B02,2026-03-03 00:00:00,082-xxx-6693
1,C00002,กิตติ,ชาย,25-34,B04,2026-01-18 00:00:00,084-xxx-3734
2,C00003,ปิยะ,หญิง,18-24,B04,2025-05-04 00:00:00,089-xxx-9286
3,C00004,ปิยะ,หญิง,18-24,B05,2026-05-05 00:00:00,083-xxx-7314
4,C00005,รัตนา,หญิง,25-34,B04,2026-07-10 00:00:00,083-xxx-5701
5,C00006,กมล,ชาย,18-24,B04,2026-08-16 00:00:00,086-xxx-1895
6,C00007,วรรณ,ชาย,45-54,B03,2026-07-17 00:00:00,087-xxx-3147
7,C00008,ภัทร,หญิง,25-34,B01,2026-04-07 00:00:00,087-xxx-9877


**สังเกต `joined_date`:** ใน Excel ดูเหมือน `2026-03-03` แต่ใน pandas เป็น `2026-03-03 00:00:00`
เพราะเซลล์ใน Excel เก็บเป็น **ค่าวันที่** (ตัวเลข เช่น 46084) ไม่ใช่ข้อความ Excel แค่ *แสดง* ให้ดูเป็นวันที่
พอ pandas อ่านเป็น str จึงได้เวลา 00:00:00 ติดมาด้วย นี่คือเหตุผลที่ต้อง profiling จากสิ่งที่ pandas อ่านได้จริง ไม่ใช่จากที่ตาเห็นใน Excel

## ขั้นที่ 2 · Profiling: หาปัญหาก่อน ยังไม่แก้

**Prompt** (ปรับจากของ sales ให้ตรงกับคอลัมน์ลูกค้า)
```
ช่วยทำ data profiling ของ DataFrame df (ข้อมูลลูกค้า) ด้วย pandas (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
- จำนวนค่าว่างและค่าไม่ซ้ำของแต่ละคอลัมน์
- จำนวนแถวที่ซ้ำกันทุกคอลัมน์ และ customer_id ที่ซ้ำ และ customer_id ที่ไม่ใช่รูปแบบ C + 5 หลัก
- จำนวน joined_date ที่ไม่ใช่รูปแบบ YYYY-MM-DD ปี ค.ศ. พร้อมตัวอย่าง และช่วงวันที่
- ค่าทั้งหมดของ gender, age_group, home_branch_id พร้อมจำนวน และจำนวนที่ไม่ใช่ค่ามาตรฐาน
- จำนวน phone ที่ไม่ใช่รูปแบบ 0XX-xxx-XXXX และจำนวนเบอร์ที่มีมากกว่า 1 คน
- จำนวนลูกค้าที่สมัครก่อนสาขาบ้านเปิด (เทียบกับ branches)
เก็บจำนวนไว้ในตัวแปร n_dup, n_dup_id, n_bad_id, n_bad_date, n_bad_gender, n_bad_age,
n_bad_branch, n_bad_phone, n_shared_phone, n_before_open
อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```

In [ ]:
# ===== Profiling ข้อมูลลูกค้า: รายงานอย่างเดียว ไม่แก้ df =====
def show(series, n=5):
    """ตัวอย่างค่าที่ไม่ซ้ำ ใช้ repr() เพื่อให้เห็นช่องว่างท้ายคำ"""
    for v in series.drop_duplicates().head(n):
        print(f"    {v!r}")

# 1) ค่าว่างและค่าไม่ซ้ำ (ช่องที่มีแต่ช่องว่างนับเป็นค่าว่างด้วย)
print("1) ค่าว่างและค่าไม่ซ้ำ")
print(pd.DataFrame({
    "ค่าว่าง": (df.apply(lambda s: s.str.strip()) == "").sum(),
    "ค่าไม่ซ้ำ": df.nunique(),
}).to_string(), "\n")

# 2) แถวซ้ำทุกคอลัมน์ / customer_id ซ้ำ / รูปแบบ id
n_dup = int(df.duplicated().sum())
n_dup_id = int(df["customer_id"].duplicated().sum())
n_bad_id = int((~df["customer_id"].str.match(r"^C\d{5}$")).sum())
print(f"2) แถวซ้ำทุกคอลัมน์: {n_dup:,} · customer_id ซ้ำ: {n_dup_id:,} · รูปแบบ id ผิด: {n_bad_id:,}\n")

# 3) joined_date ที่ไม่ใช่ YYYY-MM-DD ปี ค.ศ. (ปี พ.ศ. 25xx และ DD/MM/YYYY นับว่าผิด)
is_iso_ce = df["joined_date"].str.match(r"^(19|20)\d{2}-\d{2}-\d{2}$")
bad_date = df.loc[~is_iso_ce, "joined_date"]
n_bad_date = int(len(bad_date))
print(f"3) joined_date ไม่ใช่ YYYY-MM-DD ปี ค.ศ.: {n_bad_date:,}")
show(bad_date)
# จัดกลุ่มตามรูปแบบ (แทนตัวเลขด้วย 9) จะเห็นว่าผิดแบบไหนบ้าง เช่น มีเวลา 00:00:00 ติดมาจาก Excel
print(df["joined_date"].str.replace(r"\d", "9", regex=True).value_counts().to_string())
dates10 = df["joined_date"].str[:10]
print(f"   ช่วงวันที่ (10 ตัวแรก): {dates10.min()} ถึง {dates10.max()}\n")

# 4) ค่าหมวดหมู่ทั้งหมด
for col, std in [("gender", STD_GENDER), ("age_group", STD_AGE), ("home_branch_id", STD_BRANCH_ID)]:
    print(f"4) {col}")
    for v, c in df[col].value_counts().items():
        print(f"    {v!r:<14} {c:>6,}{'' if v in std else '   <-- ไม่มาตรฐาน'}")
n_bad_gender = int((~df["gender"].isin(STD_GENDER)).sum())
n_bad_age = int((~df["age_group"].isin(STD_AGE)).sum())
n_bad_branch = int((~df["home_branch_id"].isin(STD_BRANCH_ID)).sum())
print()

# 5) เบอร์โทร: รูปแบบที่ปกปิด และเบอร์ที่มีหลายคน
n_bad_phone = int((~df["phone"].str.match(r"^0\d{2}-xxx-\d{4}$")).sum())
phone_counts = df["phone"].value_counts()
n_shared_phone = int((phone_counts > 1).sum())
print(f"5) phone รูปแบบผิด: {n_bad_phone:,} · เบอร์ที่มีมากกว่า 1 คน: {n_shared_phone:,} เบอร์")
print("   (เบอร์ปกปิดเหลือแค่ 7 หลัก คนต่างกันจึงบังเอิญได้เบอร์เดียวกันได้ ไม่ได้แปลว่าข้อมูลซ้ำ)\n")

# 6) ตรรกะข้ามตาราง: สมัครก่อนสาขาบ้านเปิด = เป็นไปไม่ได้
#    เทียบเฉพาะ 10 ตัวแรก (YYYY-MM-DD) ทั้งสองฝั่ง ถ้ามีเวลาติดมาฝั่งเดียว
#    "2025-11-01" < "2025-11-01 00:00:00" จะเป็นจริง ทั้งที่เป็นวันเดียวกัน
opened = df["home_branch_id"].map(OPENED)
before_open = df[df["joined_date"].str[:10] < opened]
n_before_open = int(len(before_open))
print(f"6) สมัครก่อนสาขาบ้านเปิด: {n_before_open:,}")
if n_before_open:
    print(before_open.head().to_string())

print("\nสรุป")
print(pd.Series({
    "n_dup": n_dup, "n_dup_id": n_dup_id, "n_bad_id": n_bad_id, "n_bad_date": n_bad_date,
    "n_bad_gender": n_bad_gender, "n_bad_age": n_bad_age, "n_bad_branch": n_bad_branch,
    "n_bad_phone": n_bad_phone, "n_shared_phone": n_shared_phone, "n_before_open": n_before_open,
}).to_string())

1) ค่าว่างและค่าไม่ซ้ำ
                ค่าว่าง  ค่าไม่ซ้ำ
customer_id           0       3000
nickname              0         24
gender                0          3
age_group             0          6
home_branch_id        0          5
joined_date           0        530
phone                 0       2950 

2) แถวซ้ำทุกคอลัมน์: 0 · customer_id ซ้ำ: 0 · รูปแบบ id ผิด: 0

3) joined_date ไม่ใช่ YYYY-MM-DD ปี ค.ศ.: 3,000
    '2026-03-03 00:00:00'
    '2026-01-18 00:00:00'
    '2025-05-04 00:00:00'
    '2026-05-05 00:00:00'
    '2026-07-10 00:00:00'
joined_date
9999-99-99 99:99:99    3000
   ช่วงวันที่ (10 ตัวแรก): 2025-04-01 ถึง 2026-09-14

4) gender
    'หญิง'          1,652
    'ชาย'           1,232
    'ไม่ระบุ'         116
4) age_group
    '25-34'           993
    '18-24'           775
    '35-44'           577
    '45-54'           346
    '55+'             196
    'ต่ำกว่า 18'      113
4) home_branch_id
    'B01'             776
    'B05'             693
    'B02'             643
    'B

### ✔️ ผลที่ควรได้ (จาก `customers.xlsx` ใน student pack)
3,000 แถว ทุกตัวแปรเป็น **0** ยกเว้น
- `n_bad_date = 3,000` ทุกแถวมีเวลา `00:00:00` ติดมาจากค่าวันที่ของ Excel (รูปแบบวันที่เองถูกต้อง เป็น ค.ศ.)
- `n_shared_phone = 49`

**คำถามชวนคิด:** 49 เบอร์ที่ซ้ำ ควรลบไหม? ลองดู `df[df["phone"].isin(phone_counts[phone_counts > 1].index)].sort_values("phone")`
แล้วสังเกตว่าชื่อเล่น เพศ และช่วงอายุตรงกันไหม (มีเพียง 1 เบอร์ที่ชื่อเล่นและเพศตรงกัน)

## ขั้นที่ 3 · ตัดสินใจก่อนลงมือ
ดับเบิลคลิกเซลล์นี้เพื่อกรอกตาราง

| ปัญหา | ทางเลือก | ตัดสินใจ | เหตุผล |
|---|---|---|---|
| แถวซ้ำทุกคอลัมน์ | ลบ / เก็บ | | |
| customer_id ซ้ำ | เก็บแถวที่สมัครก่อน / เก็บแถวล่าสุด | | |
| joined_date ปี พ.ศ. / DD/MM/YYYY | แปลง / ลบ | | |
| gender / age_group เขียนต่างกัน | map เป็นค่ามาตรฐาน / ลบ | | |
| เบอร์เดียวกันหลายคน | เก็บ / รวมเป็นคนเดียว | | |
| สมัครก่อนสาขาบ้านเปิด | ลบ / แก้วันที่ / ถามทีม | | |

## ขั้นที่ 4 · ทำความสะอาด

ข้อมูลชุดนี้สะอาดอยู่แล้ว โค้ดนี้จึงทำหน้าที่ **ป้องกัน**: ถ้าได้ไฟล์ลูกค้าแบบ raw มา ขั้นตอนเดียวกันจะจับปัญหาได้
และจะหยุดด้วย `assert` เมื่อเจอค่าที่ไม่รู้จัก แทนที่จะปล่อยผ่านเงียบ ๆ

**ลำดับสำคัญ:** แปลงวันที่ **ก่อน** ตัด customer_id ซ้ำ เพราะการเลือก "แถวที่สมัครก่อน" ต้องเรียงวันที่ ถ้ามี DD/MM/YYYY ปนอยู่ การเรียงเป็นข้อความจะผิด

In [ ]:
# ===== ทำความสะอาดข้อมูลลูกค้า: ทำงานบน clean ห้ามแก้ df =====
clean = df.copy()
log = []

def add_log(step, rows, decision):
    log.append({"ขั้นตอน": step, "จำนวนแถวที่กระทบ": int(rows), "การตัดสินใจ": decision})

n_before = len(clean)

# 1) ตัดช่องว่างหน้า/หลังทุกคอลัมน์
stripped = clean.apply(lambda s: s.str.strip())
add_log("ตัดช่องว่างหน้า/หลัง", (stripped != clean).any(axis=1).sum(), "strip ทุกคอลัมน์")
clean = stripped

# 2) ลบแถวซ้ำทุกคอลัมน์
dup = clean.duplicated()
clean = clean[~dup]
add_log("ลบแถวซ้ำทุกคอลัมน์", dup.sum(), "ลบ เก็บแถวแรก")

# 3) joined_date เป็น YYYY-MM-DD ปี ค.ศ. (ไม่ใช้ pd.to_datetime เพราะอาจสลับวันกับเดือน)
#    รองรับเวลาที่ติดท้าย เช่น "2026-03-03 00:00:00" ซึ่งมาจากเซลล์วันที่ของ Excel
#    joined_date เป็นวันที่ล้วน จึงตัดเวลาทิ้ง
TIME = r"(?:[ T]\d{1,2}:\d{2}(?::\d{2})?)?"
def to_iso_date(s):
    m = re.match(rf"^(\d{{4}})-(\d{{1,2}})-(\d{{1,2}}){TIME}$", s)
    if m:
        y, mo, d = m.groups()
    else:
        m = re.match(rf"^(\d{{1,2}})/(\d{{1,2}})/(\d{{4}}){TIME}$", s)  # DD/MM/YYYY: วันมาก่อนเดือน
        if not m:
            return None
        d, mo, y = m.groups()
    y = int(y)
    if y > 2400:  # ปี พ.ศ.
        y -= 543
    return f"{y:04d}-{int(mo):02d}-{int(d):02d}"

new_date = clean["joined_date"].map(to_iso_date)
assert new_date.notna().all(), f"joined_date ที่แปลงไม่ได้: {clean.loc[new_date.isna(), 'joined_date'].unique()[:5]}"
add_log("แปลง joined_date เป็น ISO ค.ศ.", (new_date != clean["joined_date"]).sum(), "ตัดเวลาที่ติดมาจาก Excel แปลง พ.ศ. และ DD/MM/YYYY")
clean["joined_date"] = new_date

# 4) customer_id ซ้ำ: เก็บแถวที่สมัครก่อน (วันที่เป็น ISO แล้ว เรียงเป็นข้อความได้ถูกต้อง)
clean = clean.sort_values(["customer_id", "joined_date"])
dup_id = clean["customer_id"].duplicated()
clean = clean[~dup_id]
add_log("customer_id ซ้ำ", dup_id.sum(), "เก็บแถวที่ joined_date เก่าสุด")

# 5) ค่าหมวดหมู่: map คำที่เขียนต่างกัน แล้วตรวจว่าเหลือแค่ค่ามาตรฐาน
GENDER_MAP = {"ญ": "หญิง", "ผู้หญิง": "หญิง", "F": "หญิง", "ช": "ชาย", "ผู้ชาย": "ชาย", "M": "ชาย", "": "ไม่ระบุ"}
AGE_MAP = {"<18": "ต่ำกว่า 18", "55 ขึ้นไป": "55+"}
for col, mapping, std in [("gender", GENDER_MAP, STD_GENDER), ("age_group", AGE_MAP, STD_AGE)]:
    new = clean[col].replace(mapping)
    unknown = sorted(set(new) - set(std))
    assert not unknown, f"{col} มีค่าที่ยังไม่ได้ map: {unknown}"
    add_log(f"ปรับ {col} เป็นค่ามาตรฐาน", (new != clean[col]).sum(), f"map เหลือ {', '.join(std)}")
    clean[col] = new
bad_branch = sorted(set(clean["home_branch_id"]) - set(STD_BRANCH_ID))
assert not bad_branch, f"home_branch_id ที่ไม่มีใน branches: {bad_branch}"

# 6) เบอร์เดียวกันหลายคน: เก็บไว้ (เบอร์ปกปิดซ้ำกันได้โดยบังเอิญ ไม่ใช่หลักฐานว่าเป็นคนเดียวกัน)
add_log("เบอร์เดียวกันหลายคน", clean["phone"].duplicated(keep=False).sum(), "เก็บไว้ ไม่ใช่ข้อมูลซ้ำ")

# 7) สมัครก่อนสาขาบ้านเปิด: ไม่ลบเอง หยุดให้คนตัดสิน (อาจเป็นสาขาที่ย้าย หรือพิมพ์วันที่ผิด)
#    OPENED เป็น YYYY-MM-DD แล้ว (ตัดเวลาในขั้นที่ 1) ต้องรูปแบบเดียวกับ joined_date จึงเทียบได้ถูก
opened = clean["home_branch_id"].map(OPENED)
n_bad_logic = int((clean["joined_date"] < opened).sum())
assert n_bad_logic == 0, f"มี {n_bad_logic} คนสมัครก่อนสาขาบ้านเปิด ต้องถามทีมก่อนตัดสินใจ"
add_log("ตรวจสมัครก่อนสาขาเปิด", n_bad_logic, "ไม่พบ")

clean = clean[df.columns].sort_values("customer_id").reset_index(drop=True)  # คอลัมน์และลำดับเหมือนเดิม
cleaning_log = pd.DataFrame(log)
print(cleaning_log.to_string(index=False))
print(f"\nจำนวนแถว: ก่อน {n_before:,} → หลัง {len(clean):,} (ลดลง {n_before - len(clean):,})")

                       ขั้นตอน  จำนวนแถวที่กระทบ                                           การตัดสินใจ
          ตัดช่องว่างหน้า/หลัง                 0                                      strip ทุกคอลัมน์
            ลบแถวซ้ำทุกคอลัมน์                 0                                         ลบ เก็บแถวแรก
แปลง joined_date เป็น ISO ค.ศ.              3000     ตัดเวลาที่ติดมาจาก Excel แปลง พ.ศ. และ DD/MM/YYYY
               customer_id ซ้ำ                 0                        เก็บแถวที่ joined_date เก่าสุด
    ปรับ gender เป็นค่ามาตรฐาน                 0                          map เหลือ หญิง, ชาย, ไม่ระบุ
 ปรับ age_group เป็นค่ามาตรฐาน                 0 map เหลือ ต่ำกว่า 18, 18-24, 25-34, 35-44, 45-54, 55+
           เบอร์เดียวกันหลายคน                99                               เก็บไว้ ไม่ใช่ข้อมูลซ้ำ
         ตรวจสมัครก่อนสาขาเปิด                 0                                                 ไม่พบ

จำนวนแถว: ก่อน 3,000 → หลัง 3,000 (ลดลง 0)


## ขั้นที่ 5 · Verify
ต้องได้ ✅ ทุกข้อก่อน export

In [ ]:
def validate_customers(clean):
    ok = True
    def rep(passed, msg):
        nonlocal ok
        ok &= bool(passed)
        print(("✅ " if passed else "❌ ") + msg)

    rep(list(clean.columns) == list(df.columns), "คอลัมน์ครบและเรียงเหมือนต้นฉบับ")
    rep(clean["customer_id"].str.match(r"^C\d{5}$").all(), "customer_id เป็น C + 5 หลักทุกแถว")
    rep(not clean["customer_id"].duplicated().any(), "customer_id ไม่ซ้ำ")
    rep(clean["joined_date"].str.match(r"^20\d{2}-\d{2}-\d{2}$").all(), "joined_date เป็น YYYY-MM-DD ปี ค.ศ. ทุกแถว")
    rep(clean["gender"].isin(STD_GENDER).all(), "gender เป็นค่ามาตรฐานทุกแถว")
    rep(clean["age_group"].isin(STD_AGE).all(), "age_group เป็นค่ามาตรฐานทุกแถว")
    rep(clean["home_branch_id"].isin(STD_BRANCH_ID).all(), "home_branch_id มีอยู่ใน branches ทุกแถว")
    rep((clean["joined_date"] >= clean["home_branch_id"].map(OPENED)).all(), "ไม่มีใครสมัครก่อนสาขาบ้านเปิด")
    dropped = len(df) - len(clean)
    rep(dropped <= len(df) * 0.05, f"ตัดทิ้งไม่เกิน 5% (ตัดไป {dropped:,} แถว)")
    print("\n🎉 พร้อม export" if ok else "\nยังมีจุดต้องแก้ ลองส่งข้อความ ❌ ให้ AI ช่วยหาสาเหตุ")
    return ok

validate_customers(clean)

✅ คอลัมน์ครบและเรียงเหมือนต้นฉบับ
✅ customer_id เป็น C + 5 หลักทุกแถว
✅ customer_id ไม่ซ้ำ
✅ joined_date เป็น YYYY-MM-DD ปี ค.ศ. ทุกแถว
✅ gender เป็นค่ามาตรฐานทุกแถว
✅ age_group เป็นค่ามาตรฐานทุกแถว
✅ home_branch_id มีอยู่ใน branches ทุกแถว
✅ ไม่มีใครสมัครก่อนสาขาบ้านเปิด
✅ ตัดทิ้งไม่เกิน 5% (ตัดไป 0 แถว)

🎉 พร้อม export


True

## ขั้นที่ 5B · ตรวจข้ามตารางกับ `sales.csv`

ข้อมูลที่ผ่าน ✅ ทุกข้อในขั้นที่ 5 ถูก **รูปแบบ** แล้ว แต่อาจยังผิด **ความหมาย** เมื่อนำไปใช้คู่กับยอดขาย เช่น ซื้อก่อนวันสมัคร หรือมี `customer_id` ใน sales ที่ไม่มีในไฟล์ลูกค้า (กราฟแท็บลูกค้าจะนับหาย)

อัปโหลด `sales.csv` ที่ทำความสะอาดแล้ว (ไฟล์ใน `public/` ของ Dashboard) ขั้นนี้ใช้ `clean` เพราะ `joined_date` เป็น YYYY-MM-DD แล้ว

**Prompt**
```
ช่วยตรวจความสอดคล้องระหว่าง clean (ลูกค้า) และ sales ด้วย pandas (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
- sales มี order_id, datetime (YYYY-MM-DDTHH:MM:SS+07:00), customer_id (ค่าว่าง = บิลที่ไม่ใช่สมาชิก)
- ช่วงวันที่ของ sales เทียบกับช่วง joined_date และจำนวนลูกค้าที่สมัครหลังวันสุดท้ายของ sales
- จำนวน customer_id ใน sales (ไม่นับค่าว่าง) ที่ไม่มีใน clean
- จำนวนลูกค้าใน clean ที่ไม่เคยซื้อเลยใน sales
- จำนวนลูกค้าที่ซื้อครั้งแรกก่อน joined_date และในวันเดียวกับ joined_date
  (ใช้ datetime 10 ตัวแรก ห้ามใช้ pd.to_datetime เพราะอาจแปลงเป็น UTC ทำให้วันเลื่อน)
- สัดส่วนบิลที่เป็นของสมาชิก และจำนวนลูกค้าที่สมัครต่อเดือน
เก็บจำนวนไว้ในตัวแปร n_after_end, n_unknown_in_sales, n_never_buy, n_buy_before_join
อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```

In [ ]:
# ===== ตรวจข้ามตาราง clean × sales: รายงานอย่างเดียว ไม่แก้ข้อมูล =====
if not os.path.exists("sales.csv"):
    try:
        from google.colab import files
        print("ไม่พบ sales.csv กรุณาอัปโหลดไฟล์")
        files.upload()
    except ImportError:
        raise FileNotFoundError("ไม่พบไฟล์ sales.csv ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
sales = pd.read_csv("sales.csv", dtype=str, keep_default_na=False)

# 1) ช่วงวันที่: datetime 10 ตัวแรก = วันที่ตามเวลาไทย
sales_day = sales["datetime"].str[:10]
last_day = sales_day.max()
n_after_end = int((clean["joined_date"] > last_day).sum())
print(f"1) sales {len(sales):,} แถว ช่วง {sales_day.min()} ถึง {last_day}")
print(f"   ลูกค้าสมัคร {clean['joined_date'].min()} ถึง {clean['joined_date'].max()} · สมัครหลังวันสุดท้ายของ sales: {n_after_end:,}\n")

# 2) customer_id ที่ไม่ตรงกันระหว่างสองตาราง (ตัดบิลที่ไม่ใช่สมาชิกออกก่อน)
member = sales[sales["customer_id"] != ""]
sales_ids = set(member["customer_id"])
cust_ids = set(clean["customer_id"])
n_unknown_in_sales = len(sales_ids - cust_ids)
n_never_buy = len(cust_ids - sales_ids)
print(f"2) customer_id ใน sales {len(sales_ids):,} คน · ไม่มีใน customers: {n_unknown_in_sales:,}")
print(f"   ลูกค้าที่ไม่เคยซื้อเลย: {n_never_buy:,} จาก {len(cust_ids):,} คน ({n_never_buy / len(cust_ids):.1%})\n")

# 3) ซื้อครั้งแรกเทียบกับวันสมัคร
first_buy = sales_day[member.index].groupby(member["customer_id"]).min()
joined = clean.set_index("customer_id")["joined_date"]
common = first_buy.index.intersection(joined.index)
n_buy_before_join = int((first_buy[common] < joined[common]).sum())
n_same_day = int((first_buy[common] == joined[common]).sum())
print(f"3) ซื้อครั้งแรกก่อนวันสมัคร: {n_buy_before_join:,} · ซื้อครั้งแรกในวันที่สมัคร: {n_same_day:,}\n")

# 4) ภาพรวมเชิงธุรกิจ
n_orders = sales["order_id"].nunique()
n_member_orders = member["order_id"].nunique()
print(f"4) บิลของสมาชิก {n_member_orders:,} จาก {n_orders:,} บิล ({n_member_orders / n_orders:.1%})")
print(f"   แถว sales ที่ customer_id ว่าง (ลูกค้าทั่วไป ไม่ใช่ข้อมูลหาย): {(sales['customer_id'] == '').sum():,}")
print("   ลูกค้าสมัครต่อเดือน:")
print(clean["joined_date"].str[:7].value_counts().sort_index().to_string())

print("\nสรุป")
print(pd.Series({
    "n_after_end": n_after_end, "n_unknown_in_sales": n_unknown_in_sales,
    "n_never_buy": n_never_buy, "n_buy_before_join": n_buy_before_join,
}).to_string())

1) sales 53,057 แถว ช่วง 2025-04-01 ถึง 2026-09-20
   ลูกค้าสมัคร 2025-04-01 ถึง 2026-09-14 · สมัครหลังวันสุดท้ายของ sales: 0

2) customer_id ใน sales 2,507 คน · ไม่มีใน customers: 0
   ลูกค้าที่ไม่เคยซื้อเลย: 493 จาก 3,000 คน (16.4%)

3) ซื้อครั้งแรกก่อนวันสมัคร: 0 · ซื้อครั้งแรกในวันที่สมัคร: 97

4) บิลของสมาชิก 16,022 จาก 34,773 บิล (46.1%)
   แถว sales ที่ customer_id ว่าง (ลูกค้าทั่วไป ไม่ใช่ข้อมูลหาย): 28,494
   ลูกค้าสมัครต่อเดือน:
joined_date
2025-04    138
2025-05    140
2025-06    142
2025-07    142
2025-08    144
2025-09    148
2025-10    145
2025-11    175
2025-12    197
2026-01    190
2026-02    161
2026-03    189
2026-04    201
2026-05    200
2026-06    182
2026-07    210
2026-08    215
2026-09     81

สรุป
n_after_end             0
n_unknown_in_sales      0
n_never_buy           493
n_buy_before_join       0


### ✔️ ผลที่ควรได้ (`customers.xlsx` × `sales.csv` ที่ทำความสะอาดแล้ว)
ทุกตัวแปรเป็น **0** ยกเว้น `n_never_buy = 493` ไม่มีข้อมูลผิด แต่มีจุดที่ต้องรู้ก่อนวิเคราะห์ ไม่อย่างนั้นจะแปลผลผิด

| สิ่งที่พบ | ตัวเลข | ความหมาย |
|---|---|---|
| ลูกค้าไม่เคยซื้อเลย | 493 คน (16.4%) | ถ้าหาร "ยอดต่อสมาชิก" ด้วย 3,000 จะได้ค่าต่ำกว่าความจริง ควรเก็บไว้แต่แยกนับ |
| แถว sales ที่ `customer_id` ว่าง | 28,494 แถว | คือลูกค้าทั่วไป ไม่ใช่ข้อมูลหาย ห้ามลบทิ้ง |
| บิลของสมาชิก | 16,022 จาก 34,773 บิล (46.1%) | เป็นฐานของกราฟสัดส่วนบิลสมาชิกในแท็บลูกค้า |
| ข้อมูลสมัครจบ 14 ก.ย. 69 แต่ sales จบ 20 ก.ย. 69 | ก.ย. มีสมาชิกใหม่ 81 คน | เป็นเดือนไม่เต็ม อย่าเทียบตรง ๆ กับเดือนอื่น (ดูคำถามท้าย notebook) |
| ซื้อครั้งแรกในวันที่สมัคร | 97 คน | สมัครที่หน้าร้านตอนจ่ายเงิน สอดคล้องกับความจริง |

**คำถามชวนคิด:** ถ้า `n_unknown_in_sales` ไม่เป็น 0 ควรเพิ่มลูกค้าที่หายเข้าไปในไฟล์ลูกค้า หรือถามทีม POS ก่อน? แล้วถ้าเพิ่มเอง จะใส่ `joined_date` และ `home_branch_id` เป็นอะไร

## ขั้นที่ 6 · Export
- `customers_clean.csv`: ข้อมูลสะอาดครบทุกคอลัมน์ เก็บไว้ใช้ภายใน
- `customers_web.csv`: ตัด `nickname` และ `phone` ออก สำหรับวางใน `public/` ของ Dashboard (ไฟล์ใน `public/` ใครก็ดาวน์โหลดได้ จึงไม่ควรมีข้อมูลส่วนตัวที่กราฟไม่ได้ใช้)
- `customers_cleaning_log.csv`: หลักฐานว่าทำอะไรกับข้อมูลไปบ้าง

In [ ]:
clean.to_csv("customers_clean.csv", index=False, encoding="utf-8-sig")
clean.drop(columns=["nickname", "phone"]).to_csv("customers_web.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv("customers_cleaning_log.csv", index=False, encoding="utf-8-sig")
print("บันทึก customers_clean.csv, customers_web.csv และ customers_cleaning_log.csv แล้ว")

try:
    from google.colab import files
    for f in ["customers_clean.csv", "customers_web.csv", "customers_cleaning_log.csv"]:
        files.download(f)
except ImportError:
    pass

บันทึก customers_clean.csv, customers_web.csv และ customers_cleaning_log.csv แล้ว


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### นำไปใช้ต่อใน Dashboard
1. คัดลอก `customers_web.csv` ไปวางใน `public/` ของโปรเจกต์ แล้วเปลี่ยนชื่อเป็น `customers.csv`
2. เปิดแท็บ **ลูกค้า** (`/#customers`) กราฟ 4 กราฟและข้อความสรุปจะคำนวณใหม่จากไฟล์นี้

**คำถามชวนคิด:** ไฟล์ลูกค้ามีข้อมูลสมัครถึง 14 ก.ย. 69 แต่ sales มีถึง 20 ก.ย. 69 ถ้าคำนวณ "สมาชิกใหม่ต่อวัน" ของเดือน ก.ย. โดยหารด้วย 20 วัน จะผิดอย่างไร